# Specimen 01 — Tool-Calling Agent

Goal: give the model tools it can actually call — arithmetic, a file, a lookup — instead of just responding in text. Hand-roll the request → execute → respond cycle once before reaching for the SDK's tool runner.

In [1]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'


## 1. Define one tool by hand

A calculator function with a JSON schema (name, description, input_schema) matching what the API expects.

In [2]:
calculator_tool = {
    "name": "calculator",
    "description": "Evaluate a basic arithmetic expression involving addition, subtraction, multiplication, and division of numbers. Use this whenever the user asks for an exact numeric calculation.",
    "input_schema": {
        "type": "object",
        "properties": {
            "expression": {
                "type": "string",
                "description": "A basic arithmetic expression, e.g. '482391 * 17038'"
            }
        },
        "required": ["expression"]
    }
}

def run_calculator(expression):
    allowed = set('0123456789+-*/(). ')
    if not set(expression) <= allowed:
        raise ValueError(f'Unsupported characters in expression: {expression}')
    if len(expression) > 100:
        raise ValueError('Expression too long')
    if '**' in expression:
        raise ValueError('Exponentiation is not supported')
    return eval(expression, {"__builtins__": {}}, {})

print(run_calculator("482391 * 17038"))

8218977858


## 2. Send a request with the tool attached

Ask something the model can only answer by calling it (e.g. multiply two large, specific numbers). Check `response.stop_reason`.

In [3]:
response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculator_tool],
    messages=[{"role": "user", "content": "What is 482391 multiplied by 17038? I need the exact number."}],
    thinking={"type": "disabled"},
)

print('stop_reason:', response.stop_reason)
for block in response.content:
    print(block.type, '-', block)

stop_reason: tool_use
text - TextBlock(citations=None, text="I'll calculate that for you.", type='text')
tool_use - ToolUseBlock(id='toolu_01LAHpNvfikw8x8h4vaTd56S', caller=DirectCaller(type='direct'), input={'expression': '482391 * 17038'}, name='calculator', type='tool_use')


## 3. Execute the tool call yourself

Parse the `tool_use` block's input, run your Python function, get the real result.

In [4]:
tool_use_block = next(b for b in response.content if b.type == 'tool_use')
print('Tool requested:', tool_use_block.name)
print('Input:', tool_use_block.input)

tool_result = run_calculator(tool_use_block.input['expression'])
print('Real result:', tool_result)

Tool requested: calculator
Input: {'expression': '482391 * 17038'}
Real result: 8218977858


## 4. Send the result back

As a `tool_result` block in a new user turn, then read the model's final answer.

In [5]:
messages = [
    {"role": "user", "content": "What is 482391 multiplied by 17038? I need the exact number."},
    {"role": "assistant", "content": response.content},
    {"role": "user", "content": [
        {"type": "tool_result", "tool_use_id": tool_use_block.id, "content": str(tool_result)}
    ]},
]

final_response = client.messages.create(
    model=MODEL,
    max_tokens=300,
    tools=[calculator_tool],
    messages=messages,
    thinking={"type": "disabled"},
)

final_text = ''.join(b.text for b in final_response.content if b.type == 'text')
print(final_text)

482391 × 17038 = **8,218,977,858**


## 5. Add a second tool

A file reader or a small search function over a fixed local dataset (not real web search yet). Confirm the model picks the right tool for the right question.

In [6]:
local_dataset = {
    "population_tokyo": "37.4 million (2024 metro estimate)",
    "population_paris": "11.2 million (2024 metro estimate)",
    "population_cairo": "22.2 million (2024 metro estimate)",
}

lookup_tool = {
    "name": "lookup_fact",
    "description": "Look up a known fact from a small local dataset by key. Use this instead of guessing when the user asks about a specific stored fact like a city's population.",
    "input_schema": {
        "type": "object",
        "properties": {
            "key": {
                "type": "string",
                "description": "The dataset key to look up, e.g. 'population_tokyo'"
            }
        },
        "required": ["key"]
    }
}

def run_lookup(key):
    return local_dataset.get(key, f"No entry found for '{key}'")

def call_with_tools(user_message, tools, tool_functions, max_tokens=400, max_steps=8):
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
    step = 1

    while response.stop_reason == 'tool_use':
        if step >= max_steps:
            raise RuntimeError(f'Hit max_steps={max_steps} without finishing')
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                fn = tool_functions[block.name]
                if block.name == 'calculator':
                    result = fn(block.input['expression'])
                elif block.name == 'lookup_fact':
                    result = fn(block.input['key'])
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
        step += 1

    return ''.join(b.text for b in response.content if b.type == 'text')

tools = [calculator_tool, lookup_tool]
tool_functions = {"calculator": run_calculator, "lookup_fact": run_lookup}

print(call_with_tools("What's the population of Tokyo?", tools, tool_functions))
print(call_with_tools("What is 913 times 27?", tools, tool_functions))

The population of Tokyo is **37.4 million** — that's the 2024 estimate for the greater Tokyo metropolitan area, which makes it the largest urban area in the world.

Note that Tokyo's population figures vary a lot depending on how you draw the boundaries: the metro area figure above is much larger than the population of Tokyo's core 23 special wards or the Tokyo prefecture alone. Let me know if you'd like me to look up any related figures.


913 × 27 = **24,651**


## 6. Handle a tool that fails

Make one tool raise/return an error on purpose, return it with `is_error: true`. Confirm the model adapts instead of your loop crashing.

In [7]:
def run_calculator_flaky(expression):
    if 'error' in expression.lower():
        raise ValueError("Simulated calculator failure: could not parse expression")
    return run_calculator(expression)

def call_with_tools_safe(user_message, tools, tool_functions, max_tokens=400, max_steps=8):
    messages = [{"role": "user", "content": user_message}]
    response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
    step = 1

    while response.stop_reason == 'tool_use':
        if step >= max_steps:
            raise RuntimeError(f'Hit max_steps={max_steps} without finishing')
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == 'tool_use':
                fn = tool_functions[block.name]
                try:
                    if block.name == 'calculator':
                        result = fn(block.input['expression'])
                    elif block.name == 'lookup_fact':
                        result = fn(block.input['key'])
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(result)})
                except Exception as e:
                    tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(e), "is_error": True})
        messages.append({"role": "user", "content": tool_results})
        response = client.messages.create(model=MODEL, max_tokens=max_tokens, tools=tools, messages=messages, thinking={"type": "disabled"})
        step += 1

    return ''.join(b.text for b in response.content if b.type == 'text')

broken_tools = [calculator_tool, lookup_tool]
broken_functions = {"calculator": run_calculator_flaky, "lookup_fact": run_lookup}

print(call_with_tools_safe("Please calculate this: error_case + 5", broken_tools, broken_functions))

I can't calculate that — `error_case` isn't a number, so `error_case + 5` isn't a valid arithmetic expression.

A couple of possibilities:

- **If `error_case` is meant to be a stored fact**, I can look it up and then add 5 — just confirm the exact key name (e.g. `error_case`) and I'll try it.
- **If it's a placeholder or typo**, let me know the actual number you had in mind and I'll do the arithmetic.
